# 02 — Contrats de données — ce que le projet refuse d'ingérer

**Projet** : Routage de tickets de support par TF-IDF — la référence explicable

## Objectifs pédagogiques

1. Lire les contrats Pandera du corpus et de la table de prédictions.
1. Casser chaque règle volontairement et lire le message d'erreur produit.
1. Comprendre le mode *lazy* : collecter toutes les violations en un seul passage.
1. Relier chaque colonne du contrat à l'usage qui en est fait plus loin.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (320 tickets) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 320

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(CONFIG.data.target or "label")
LABELS = [
    "facturation",
    "livraison",
    "produit_defectueux",
    "remboursement",
    "compte_client",
    "autre",
]
STYLES = [
    "canonique",
    "paraphrase",
    "bruite",
]

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Classes           : {len(LABELS)} — {', '.join(LABELS)}")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticTicketGenerator
from src.data.loaders import TextLabelLoader

LOADER = TextLabelLoader(
    PATHS,
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} tickets, graine {METADATA.get('seed')}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticTicketGenerator.from_config(CONFIG.data, seed=CONFIG.seed).generate()
    DOCUMENTS, METADATA = bundle.documents, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} tickets généré et persisté")

TRAIN = LOADER.split("train", frame=DOCUMENTS)
VALIDATION = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION = LOADER.split("calibration", frame=DOCUMENTS)
TEST = LOADER.split("test", frame=DOCUMENTS)
# La calibration sert à **régler** (seuil d'automatisation), la validation à **choisir**
# (algorithme, rééquilibrage), et le test à **juger** — une seule fois, à la fin.
print(
    "splits :",
    {
        "train": len(TRAIN),
        "val": len(VALIDATION),
        "calibration": len(CALIBRATION),
        "test": len(TEST),
    },
)

LABEL_COUNTS = DOCUMENTS[TARGET_COLUMN].value_counts()
MAJORITY_LABEL = str(LABEL_COUNTS.index[0])
MINORITY_LABEL = str(LABEL_COUNTS.index[-1])
print(
    f"classe majoritaire : {MAJORITY_LABEL} ({LABEL_COUNTS.iloc[0] / len(DOCUMENTS):.1%}) "
    f"| classe minoritaire : {MINORITY_LABEL} ({LABEL_COUNTS.iloc[-1] / len(DOCUMENTS):.1%})"
)

## 2. Le contrat du corpus

Le corpus est décrit une seule fois, dans `src/data/schemas.py`, et ce contrat sert à la génération, au chargement et à l'écriture. Un schéma qui ne sert qu'à documenter ne protège personne : ici, tout ce qui entre dans le projet passe par lui.

In [ ]:
from src.data.schemas import (
    LABELS as DECLARED_LABELS,
)
from src.data.schemas import (
    probability_columns,
    validate_predictions,
    validate_tickets,
)

validated = validate_tickets(DOCUMENTS)
print(f"corpus validé : {len(validated)} lignes, {len(validated.columns)} colonnes")
print("colonnes :", ", ".join(str(name) for name in validated.columns))
print("classes déclarées :", DECLARED_LABELS)
print("colonnes de probabilités :", probability_columns())

In [ ]:
from pandera.errors import SchemaError, SchemaErrors


def expect_failure(label: str, frame: pd.DataFrame, validator) -> None:
    """Montre le refus du contrat : une règle qu'on n'a jamais vue échouer n'est pas une règle."""
    try:
        validator(frame)
    except (SchemaError, SchemaErrors) as error:
        first = str(error).strip().splitlines()[0]
        print(f"[refusé] {label}\n         {first[:150]}")
    else:
        raise AssertionError(f"{label} aurait dû être refusé par le contrat")


sabotaged = DOCUMENTS.copy()
sabotaged.loc[sabotaged.index[1], "doc_id"] = sabotaged.loc[sabotaged.index[0], "doc_id"]
expect_failure("identifiant dupliqué", sabotaged, validate_tickets)

sabotaged = DOCUMENTS.copy()
sabotaged.loc[sabotaged.index[0], TARGET_COLUMN] = "reclamation_inconnue"
expect_failure("classe hors nomenclature", sabotaged, validate_tickets)

sabotaged = DOCUMENTS.copy()
sabotaged.loc[sabotaged.index[0], "text"] = "trop court"
expect_failure("texte tronqué", sabotaged, validate_tickets)

sabotaged = DOCUMENTS.copy()
sabotaged.loc[sabotaged.index[0], "n_tokens"] = -3
expect_failure("longueur négative", sabotaged, validate_tickets)

**Ce qu'il faut retenir**

- Chaque refus nomme la colonne et la règle : le message s'adresse à la personne qui devra corriger la donnée, pas au développeur du schéma.
- L'identifiant dupliqué est refusé parce que toute la reproductibilité du projet repose sur `doc_id` (jointures entre corpus, prédictions et rapport).
- Une classe inconnue est refusée **avant** l'entraînement : c'est le genre d'erreur qui produirait sinon un modèle silencieusement amputé d'une catégorie.

## 3. Le contrat des prédictions

La table de prédictions est ce qui sort du projet : elle est jointe au rapport et relue par l'équipe support. Son contrat est donc aussi strict que celui du corpus — avec une différence de taille : `expected_label` est **nullable**, parce qu'à l'inférence la vérité terrain n'existe pas.

In [ ]:
from src.models import build_model

model = build_model(CONFIG)
model.fit(TRAIN, target=TARGET_COLUMN)
scored = model.predict_frame(VALIDATION, text_column=TEXT_COLUMN)
scored.insert(0, "doc_id", VALIDATION["doc_id"].to_numpy())
scored.insert(1, "text", VALIDATION[TEXT_COLUMN].astype(str).to_numpy())
scored["expected_label"] = VALIDATION[TARGET_COLUMN].astype(str).to_numpy()
scored["correct"] = (scored["prediction"] == scored["expected_label"]).astype(int)
scored["latency_ms"] = 0.0

predictions = validate_predictions(scored)
print(f"prédictions validées : {len(predictions)} lignes")
display(predictions[["doc_id", "expected_label", "prediction", "confidence", "correct"]].head())
print(
    "probabilités d'une ligne :", predictions.loc[0, list(probability_columns())].round(4).to_dict()
)

In [ ]:
broken = scored.copy()
broken.loc[broken.index[0], probability_columns()[0]] = 0.99
expect_failure("probabilités qui ne somment plus à 1", broken, validate_predictions)

broken = scored.copy()
broken.loc[broken.index[0], "confidence"] = 1.4
expect_failure("confiance hors de [0, 1]", broken, validate_predictions)

broken = scored.copy()
broken.loc[broken.index[0], "prediction"] = "colis_perdu"
expect_failure("libellé inconnu dans la prédiction", broken, validate_predictions)

broken = scored.drop(columns=["latency_ms"])
expect_failure("colonne obligatoire absente", broken, validate_predictions)

**Ce qu'il faut retenir**

- Le contrôle de somme des probabilités est fait **au niveau du schéma** : une table de prédictions qui n'est plus une distribution est refusée, pas arrondie en silence.
- Le contrat est strict sur les colonnes (`strict = True`) : ajouter une colonne implicite dans un notebook ne passera pas la porte du rapport.
- Le contrôle « latence ≥ 0 » empêche de publier une mesure de performance négative, qui est toujours le signe d'une horloge mal instrumentée.

## 4. Validation paresseuse : toutes les erreurs d'un seul passage

En mode `lazy`, Pandera collecte **toutes** les violations avant de lever. C'est ce qu'on veut sur un rapport de qualité de données : corriger dix lignes en un passage coûte moins cher que dix allers-retours.

In [ ]:
lazy_frame = DOCUMENTS.copy()
lazy_frame.loc[lazy_frame.index[0], TARGET_COLUMN] = "hors_liste"
lazy_frame.loc[lazy_frame.index[1], "n_tokens"] = -1
lazy_frame.loc[lazy_frame.index[2], "text"] = "court"

try:
    validate_tickets(lazy_frame, lazy=True)
except SchemaErrors as errors:
    failures = errors.failure_cases
    print(f"{len(failures)} violations collectées d'un coup")
    display(failures[["column", "check", "failure_case"]].head(8))

In [ ]:
validation_node = dict(node(CONFIG, "data").get("validation", {}))
print("politique de validation du projet :", validation_node)
print("rappel : `raw` couvre le chargement du corpus, `processed` les tables intermédiaires,")
print(
    "         `inference` les textes arrivant en production, `strict`/`lazy` règlent la sévérité."
)

**Ce qu'il faut retenir**

- Le mode paresseux sert aux rapports qualité ; le mode strict sert à la CI, où l'on veut échouer vite.
- Les quatre interrupteurs (`raw`, `processed`, `inference`, `strict`/`lazy`) sont dans la configuration Hydra : aucune validation ne s'active par surprise.
- Un contrat qu'on peut désactiver est un contrat dont il faut lire la configuration — c'est exactement ce que fait la cellule précédente.